In [19]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Modelo principal proposto
### Desfecho
"MG"
"LEVE"

### Variáveis independentes
"ACESSO_LOCAL",    
"TEMPO",  
"INCID_MEDIA_GERAL",  
"PROP_POP_0A10",  
"PROP_POP_60MAIS",  
"LOG_POP",  
"PROP_CLASS_IGN"


### O que queremos descobrir?
* Municípios com maior tempo/distância até o PESA apresentam maior chance de ocorrência de casos moderados/graves?

Controlando por:

- incidência;
- estrutura etária;
- porte populacional;
- qualidade do preenchimento.

In [20]:
df = pd.read_csv(filepath_or_buffer='Dados-modelo/df_preliminar.csv', sep=';')
df = df[df['MG'] + df['LEVE'] > 0].copy() # Para modelo sem resultados zerados para numero de casos
df = df[df['ACESSO_LOCAL']==0].copy()

In [21]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'REGIAO',
       'PESA', 'MUNI_REFERENCIADO', 'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI',
       'LAT_PESA', 'LON_PESA', 'DISTANCIA', 'TEMPO', 'IBGE', 'MUNI_NOME_x',
       'POP10', 'POP12', 'POP11A59', 'POP60', 'POP_GERAL', 'MACRO_CODIGO',
       'MACRO_NOME', 'DRS_CODIGO', 'DRS_NOME', 'CIR_CODIGO', 'CIR_NOME',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_12A', 'TOTAL_11A59', 'TOTAL_60',
       'LEVE', 'MG', 'LEVE_10', 'MG_10', 'LEVE_12', 'MG_12', 'LEVE_11A59',
       'MG_11A59', 'LEVE_60', 'MG_60', 'TOTAL_AMPOLAS', 'CAT_TEMPO',
       'PROP_POP_0A10', 'PROP_POP_60MAIS', 'LOG_POP', 'PROP_MG',
       'INCID_MEDIA_GERAL', 'INCID_0A10', 'INCID_60MAIS', 'PROP_CASOS_0A10',
       'PROP_CASOS_60MAIS', 'TAXA_MG_100MIL', 'CLASS_IG', 'SORO_IG', 'EVOL_IG',
       'PROP_CLASS_IGN', 'PROP_SORO_IGN', 'PROP_EVOL_IGN', 'BAIXO_N'],
      dtype='str')

In [22]:
# Criar logaritmo da população (caso ainda não exista)
df["LOG_POP"] = np.log(df["POP_GERAL"])

# Ajustar o modelo binomial com correção para superdispersão
modelo_binomial = smf.glm(

    formula="""
        MG + LEVE ~
        TEMPO +
        INCID_MEDIA_GERAL +
        PROP_POP_0A10 +
        PROP_POP_60MAIS +
        LOG_POP +
        PROP_CLASS_IGN
    """,

    data=df,

    family=sm.families.Binomial()

).fit(scale="X2")

# Resumo do modelo
print(modelo_binomial.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         ['MG', 'LEVE']   No. Observations:                  440
Model:                            GLM   Df Residuals:                      433
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                         0.17387
Method:                          IRLS   Log-Likelihood:                -1221.0
Date:                Sat, 18 Jul 2026   Deviance:                       1277.6
Time:                        00:19:25   Pearson chi2:                 1.50e+03
No. Iterations:                     8   Pseudo R-squ. (CS):             0.4138
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -2.1251      0.21

In [23]:
or_modelo = np.exp(modelo_binomial.params)

ic = np.exp(modelo_binomial.conf_int())

resultado = pd.concat(
    [or_modelo, ic],
    axis=1
)

resultado.columns = [
    "OR",
    "IC95_inf",
    "IC95_sup"
]

print(resultado)

# Interpretação: Cada minuto adicional até o PESA aumenta em 1,3% a chance de um caso ser moderado/grave.

                           OR    IC95_inf     IC95_sup
Intercept            0.119424    0.077974     0.182908
TEMPO                0.993763    0.992062     0.995467
INCID_MEDIA_GERAL    0.998819    0.998731     0.998907
PROP_POP_0A10      788.069012  179.987265  3450.537271
PROP_POP_60MAIS      0.581027    0.231404     1.458885
LOG_POP              0.910974    0.893624     0.928660
PROP_CLASS_IGN       1.629161    1.306854     2.030958


In [24]:
# Verificar multicolinearidade
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[
    [
        'TEMPO',
        'INCID_MEDIA_GERAL',
        'PROP_POP_0A10',
        'PROP_POP_60MAIS',
        'LOG_POP',
        'PROP_CLASS_IGN'
    ]
]

X = sm.add_constant(X)

vif = pd.DataFrame()

vif["Variavel"] = X.columns

vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif)

            Variavel         VIF
0              const  407.845199
1              TEMPO    1.094491
2  INCID_MEDIA_GERAL    1.218976
3      PROP_POP_0A10    1.991901
4    PROP_POP_60MAIS    2.446307
5            LOG_POP    1.489583
6     PROP_CLASS_IGN    1.063344


In [25]:
# Municípios com acesso local apresentam menos MG do que municípios a mais de 60 minutos?
modelo_cat = smf.glm(

    formula="""
        MG + LEVE ~
        C(CAT_TEMPO)
    """,

    data=df,

    family=sm.families.Binomial()

).fit()

print(modelo_cat.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         ['MG', 'LEVE']   No. Observations:                  440
Model:                            GLM   Df Residuals:                      437
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1336.6
Date:                Sat, 18 Jul 2026   Deviance:                       1508.8
Time:                        00:19:25   Pearson chi2:                 1.94e+03
No. Iterations:                     6   Pseudo R-squ. (CS):           0.008472
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [26]:
# Razao aproximou-se de 1?
print(
    modelo_binomial.pearson_chi2 /
    modelo_binomial.df_resid
)

3.4527122676111346


In [29]:
df.groupby("CAT_TEMPO").agg(
    {
        "PROP_MG": ["mean", "median"],
        "INCID_MEDIA_GERAL": ["mean", "median"],
        "TOTAL_CASOS": ["mean", "median"]
    }
)

PROP_MG        INCID_MEDIA_GERAL          TOTAL_CASOS       
               mean median              mean   median        mean median
CAT_TEMPO                                                               
31–60 min  0.094266  0.060        200.178252  141.520   72.286713   46.0
>60 min    0.105000  0.025         49.736667   31.075   22.833333   16.5
≤30 min    0.094502  0.060        174.581993   87.820   88.529210   47.0